# 02 — Reproducible model fitting / reserved audit

Fit candidate LightGBM and CatBoost pipelines with a real ColumnTransformer. Calibration chooses a blend and per-grade offsets; held-out AUDIT is not used in model selection. **Use `--quick` only for a smoke test.**

**Status:** pipeline ini belum dilatih ulang pada dataset penuh dalam pembaruan ini. Hasil notebook 04 berasal dari eksperimen berbeda. AUDIT disisihkan dalam run ini; dataset pernah dipakai untuk tuning sebelumnya sehingga bukan test independen yang benar-benar baru. Gunakan direktori output terpisah agar incumbent tetap tersimpan.

In [ ]:
import sys, argparse, json
from pathlib import Path
ROOT = Path.cwd() if Path('scripts').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from scripts.preprocess import read_competition_data, parse_grades
from scripts.train import train
train_df, test_df, sample = read_competition_data(ROOT/'data'/'structural-damage-classification.zip')
print('Training', train_df.shape, 'Test', test_df.shape)
print('Target counts', dict(zip(*__import__('numpy').unique(parse_grades(train_df.damage_grade), return_counts=True))))

In [ ]:
# Full run. On large datasets this can require substantial CPU and memory.
settings = argparse.Namespace(data=str(ROOT/'data'/'structural-damage-classification.zip'), output=str(ROOT/'artifacts/notebook02_modular'), seed=42, quick=False)
selection = train(settings)
print('Selected model:', selection['name'])
print('Calibration Macro F1:', selection['cal_f1'])
print('Reserved audit Macro F1:', selection['audit_f1'])
print('Historical public score is NOT a directly comparable audit score.')

In [ ]:
from IPython.display import display
audit = json.loads((ROOT/'artifacts/notebook02_modular'/'audit_metrics.json').read_text())
print('Confusion matrix (true rows, predicted columns)')
display(__import__('pandas').DataFrame(audit['confusion_matrix'], index=range(1,6), columns=range(1,6)))
display(__import__('pandas').DataFrame(audit['classification_report']).T)
print('Important: submit to Kaggle separately to obtain an actual public score.')